In [4]:
import undetected_chromedriver as uc
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import TimeoutException, NoSuchElementException
from bs4 import BeautifulSoup
import pandas as pd
import time
import random
from datetime import datetime

# ─── CONFIG ───────────────────────────────────────────────────────────────────
INPUT_FILE  = "../data/cinema_streaming_data.csv"
OUTPUT_FILE = "../data/cross_validation_data.csv"
TCONST_COL  = "tconst"
# ──────────────────────────────────────────────────────────────────────────────

def parse_imdb_date(date_str):
    """Convert IMDb date string (e.g. 'December 15, 2011') to datetime object."""
    try:
        return datetime.strptime(date_str.strip(), "%B %d, %Y")
    except ValueError:
        return None

def start_browser():
    """Launch an undetected Chrome instance."""
    options = uc.ChromeOptions()
    options.add_argument("--lang=en-US")
    options.add_argument("--no-sandbox")
    options.add_argument("--disable-dev-shm-usage")
    # Remove the line below to watch the browser work (useful for debugging)
    # options.add_argument("--headless=new")
    driver = uc.Chrome(options=options, version_main=146)
    driver.set_window_size(1280, 900)
    return driver

def click_see_more(driver):
    """Click ALL expand buttons on the page (covers both '50 more' and 'See all' variants)."""
    try:
        buttons = driver.find_elements(By.CSS_SELECTOR, "button.ipc-see-more__button")
        for btn in buttons:
            try:
                driver.execute_script("arguments[0].click();", btn)
                time.sleep(0.3)
            except Exception:
                pass  # button may have disappeared after a previous click
    except Exception:
        pass

def get_us_wide_release(driver, tconst):
    """
    Navigate to the IMDb releaseinfo page for a given tconst,
    expand all dates, and return the earliest US-wide release date
    (i.e. no qualifier in parentheses).
    """
    url = f"https://www.imdb.com/title/{tconst}/releaseinfo/"

    try:
        driver.get(url)

        # Wait until at least one release date list item is present
        WebDriverWait(driver, 10).until(
            EC.presence_of_element_located((By.CSS_SELECTOR, "li[data-testid='list-item']"))
        )
    except TimeoutException:
        # Fallback: wait a bit and try parsing whatever loaded
        time.sleep(0.3)

    # Click "see more" button to expand all dates
    click_see_more(driver)

    # Parse the fully rendered page
    soup = BeautifulSoup(driver.page_source, "html.parser")

    # ── Find all release date list items ──────────────────────────────────────
    # IMDb uses data-testid="list-item" on each release date row
    items = soup.find_all("li", attrs={"data-testid": "list-item"})

    # Fallback: try by class if data-testid not present
    if not items:
        items = soup.find_all(
            "li",
            attrs={"role": "presentation",
                   "class": lambda c: c and "ipc-metadata-list__item" in " ".join(c)}
        )

    us_wide_dates = []
    us_all_dates  = []

    for item in items:
        # ── Country ───────────────────────────────────────────────────────────
        # Country label is an <a> with aria-label="United States"
        country_tag = item.find("a", attrs={"aria-label": "United States"})
        if not country_tag:
            # Try plain text match as fallback
            label = item.find(class_=lambda c: c and "ipc-metadata-list-item__label" in " ".join(c) if c else False)
            if not label or "United States" not in label.get_text():
                continue

        # ── Date + qualifier ──────────────────────────────────────────────────
        # Each row can have multiple sub-items (one per release event)
        sub_items = item.find_all("li", class_=lambda c: c and "ipc-metadata-list-item__list-content-item" in " ".join(c) if c else False)

        # If no sub-items found, try reading spans directly from the item
        if not sub_items:
            sub_items = [item]

        for si in sub_items:
            spans = si.find_all("span")
            if not spans:
                continue

            date_raw  = spans[0].get_text(strip=True)
            qualifier = spans[1].get_text(strip=True) if len(spans) > 1 else ""

            parsed = parse_imdb_date(date_raw)
            if not parsed:
                continue

            us_all_dates.append((parsed, date_raw, qualifier))

            # No qualifier = wide national release
            if qualifier == "":
                us_wide_dates.append((parsed, date_raw))

    # ── Select best date ──────────────────────────────────────────────────────
    if us_wide_dates:
        best = min(us_wide_dates, key=lambda x: x[0])
        print(f"  [OK]       {tconst} → {best[1]}")
        return {"tconst": tconst, "us_release_date": best[0].strftime("%Y-%m-%d"),
                "us_release_date_raw": best[1], "notes": "wide"}

    elif us_all_dates:
        best = min(us_all_dates, key=lambda x: x[0])
        print(f"  [FALLBACK] {tconst} → {best[1]} (qualifier: '{best[2]}')")
        return {"tconst": tconst, "us_release_date": best[0].strftime("%Y-%m-%d"),
                "us_release_date_raw": best[1], "notes": f"fallback: {best[2]}"}

    else:
        print(f"  [MISSING]  {tconst}")
        return {"tconst": tconst, "us_release_date": None,
                "us_release_date_raw": None, "notes": "no_us_date"}


# ─── MAIN ─────────────────────────────────────────────────────────────────────
def main():
    df      = pd.read_csv(INPUT_FILE)
    tconsts = df[TCONST_COL].dropna().unique().tolist()

    print(f"Starting scrape for {len(tconsts)} movies...\n")

    driver  = start_browser()
    results = []

    try:
        for i, tconst in enumerate(tconsts, start=1):
            print(f"[{i}/{len(tconsts)}] {tconst}")
            result = get_us_wide_release(driver, tconst)
            results.append(result)

            # Random sleep between pages to avoid triggering rate limits
            time.sleep(random.uniform(0.3, 0.5))

            # Checkpoint save every 50 movies
            if i % 50 == 0:
                pd.DataFrame(results).to_csv(OUTPUT_FILE, index=False)
                print(f"\n  [CHECKPOINT] Saved {i} results → {OUTPUT_FILE}\n")

    finally:
        driver.quit()

    # Final save
    df_out = pd.DataFrame(results)
    df_out.to_csv(OUTPUT_FILE, index=False)

    # Summary
    wide     = (df_out["notes"] == "wide").sum()
    fallback = df_out["notes"].str.startswith("fallback", na=False).sum()
    missing  = df_out["us_release_date"].isna().sum()

    print(f"\n{'─'*50}")
    print(f"Done. Saved to {OUTPUT_FILE}")
    print(f"  Wide release found : {wide}")
    print(f"  Fallback used      : {fallback}")
    print(f"  No date found      : {missing}")
    print(f"  Total              : {len(df_out)}")

if __name__ == "__main__":
    main()

Starting scrape for 1143 movies...

[1/1143] tt0359950
  [OK]       tt0359950 → December 25, 2013
[2/1143] tt0365907
  [OK]       tt0365907 → September 19, 2014
[3/1143] tt0377981
  [OK]       tt0377981 → February 11, 2011
[4/1143] tt0383010
  [OK]       tt0383010 → April 13, 2012
[5/1143] tt0385887
  [OK]       tt0385887 → November 1, 2019
[6/1143] tt0401729
  [OK]       tt0401729 → March 9, 2012
[7/1143] tt0409847
  [OK]       tt0409847 → July 29, 2011
[8/1143] tt0427152
  [OK]       tt0427152 → July 30, 2010
[9/1143] tt0429493
  [OK]       tt0429493 → June 11, 2010
[10/1143] tt0431021
  [OK]       tt0431021 → August 31, 2012
[11/1143] tt0433035
  [OK]       tt0433035 → October 7, 2011
[12/1143] tt0435761
  [OK]       tt0435761 → June 18, 2010
[13/1143] tt0437086
  [OK]       tt0437086 → February 14, 2019
[14/1143] tt0443272
  [OK]       tt0443272 → November 16, 2012
[15/1143] tt0443465
  [FALLBACK] tt0443465 → May 22, 2015 (qualifier: '(Seattle International Film Festival)')
[16/114